# Module 4.3: Regime-Aware Trading Strategy Backtest

## Learning Objectives
By completing this notebook, you will:
1. Design regime-conditional trading strategies
2. Implement realistic backtesting framework
3. Account for transaction costs and slippage
4. Evaluate risk-adjusted performance metrics
5. Compare regime-aware vs. static strategies

## Prerequisites
- Market regime detection (Module 4.1-4.2)
- Portfolio management basics

## Estimated Time: 55 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Strategy Design

Regime-aware allocation:
- **Bull**: 100% equities
- **Bear**: 30% equities, 70% bonds
- **Sideways**: 60% equities, 40% bonds

In [ ]:
def generate_multi_asset_data(T=2520):
    np.random.seed(42)
    # 3 regimes
    A = np.array([[0.95,0.03,0.02],[0.05,0.90,0.05],[0.10,0.05,0.85]])
    pi = np.array([0.5,0.2,0.3])
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(3, p=pi)
    for t in range(1,T): states[t] = np.random.choice(3, p=A[states[t-1]])
    # Equity returns
    equity_params = {0:{'mu':0.0008,'sig':0.01}, 1:{'mu':-0.0012,'sig':0.025}, 2:{'mu':0.0001,'sig':0.008}}
    # Bond returns
    bond_params = {0:{'mu':0.0002,'sig':0.003}, 1:{'mu':0.0003,'sig':0.004}, 2:{'mu':0.0002,'sig':0.003}}
    equity_ret = np.array([np.random.normal(equity_params[s]['mu'], equity_params[s]['sig']) for s in states])
    bond_ret = np.array([np.random.normal(bond_params[s]['mu'], bond_params[s]['sig']) for s in states])
    dates = pd.date_range('2015-01-01', periods=T, freq='B')
    return pd.DataFrame({'Date':dates, 'Equity':equity_ret, 'Bond':bond_ret, 'State':states})

data = generate_multi_asset_data()
print(data.head())

## 2. Backtest Implementation

In [ ]:
class RegimeStrategy:
    def __init__(self, allocations, transaction_cost=0.001):
        self.allocations = allocations
        self.tc = transaction_cost
    
    def backtest(self, returns, detected_states):
        T = len(returns)
        weights = np.array([self.allocations[s] for s in detected_states])
        portfolio_ret = (weights * returns).sum(axis=1)
        # Transaction costs
        weight_changes = np.abs(np.diff(weights, axis=0, prepend=weights[0:1]))
        costs = weight_changes.sum(axis=1) * self.tc
        net_ret = portfolio_ret - costs
        cum_ret = np.cumprod(1 + net_ret) - 1
        sharpe = net_ret.mean() / net_ret.std() * np.sqrt(252)
        return {'cumulative': cum_ret, 'sharpe': sharpe, 'total_return': cum_ret[-1]}

allocations = {0: np.array([1.0, 0.0]), 1: np.array([0.3, 0.7]), 2: np.array([0.6, 0.4])}
strategy = RegimeStrategy(allocations)
print('Strategy initialized')

## Exercise 1: Full Strategy Backtest

**Task**: Fit HMM, decode regimes, run backtest

**Expected Output**: Performance metrics

In [ ]:
# YOUR CODE HERE
def run_full_backtest(data, allocations):
    # 1. Fit HMM to equity returns
    # 2. Decode states
    # 3. Run backtest
    return None

results = run_full_backtest(data, allocations)

In [ ]:
def test_exercise_1():
    assert results is not None
    print('✅ Exercise 1 passed!')
test_exercise_1()

## Exercise 2: Compare Multiple Strategies

In [ ]:
# YOUR CODE HERE
def compare_strategies(data, strategy_list):
    return None

comparison = compare_strategies(data, [allocations])

In [ ]:
def test_exercise_2():
    assert comparison is not None
    print('✅ Exercise 2 passed!')
test_exercise_2()

## Exercise 3: Transaction Cost Sensitivity

In [ ]:
# YOUR CODE HERE
def analyze_cost_sensitivity(data, allocations, cost_range):
    return None

sensitivity = analyze_cost_sensitivity(data, allocations, [0, 0.001, 0.005, 0.01])

In [ ]:
def test_exercise_3():
    assert sensitivity is not None
    print('✅ Exercise 3 passed!')
test_exercise_3()

## Summary

### Key Takeaways
1. Regime-aware strategies can outperform static allocation
2. Transaction costs matter - frequent rebalancing hurts
3. Risk-adjusted returns improve with regime detection
4. Robustness to regime misclassification is crucial
5. Out-of-sample testing essential

---